# MarsLandformNet V4b — FiLM Conditioning (MOLA modulates HiRISE)

**Same data as V4, different MOLA fusion architecture.**

| | V4 (concat) | V4b (FiLM) |
|---|---|---|
| MOLA fusion | Concatenate with CLS token | **FiLM: MOLA generates γ,β to modulate CLS** |
| Head input | 832-dim (768+64) | **768-dim** (modulated CLS token) |
| Idea | MLP learns interactions from flat vector | **MOLA tells model HOW to read visual features** |

**FiLM** (Feature-wise Linear Modulation):
```
γ, β = Linear(MOLA)      # MOLA generates per-feature scale and shift
CLS_modulated = γ * CLS + β   # terrain context conditions visual interpretation
```

**Why**: Same surface texture at different elevations/slopes may indicate different landforms.
MOLA should tell the model *how* to interpret visual features, not just *what* to concatenate.

**Target**: Macro F1 ≥ 0.80

## 0. Setup

In [ ]:
!pip install -q transformers peft accelerate timm pillow scikit-learn matplotlib seaborn

In [ ]:
import os
import json
import copy
import time
import math
import random
import warnings
from pathlib import Path
from collections import Counter

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torch.cuda.amp import autocast, GradScaler
from PIL import Image
from sklearn.metrics import f1_score, classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings('ignore')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'GPU: {gpu_name} ({gpu_mem:.1f}GB)')
else:
    print('WARNING: No GPU! Runtime > Change runtime type > T4 GPU')
print(f'PyTorch: {torch.__version__}, Device: {device}')

## 1. Download Data

In [ ]:
DATA_DIR = Path('/content/v4_data')
DATA_DIR.mkdir(exist_ok=True)

GITHUB_TOKEN = ''  # Set if repo is private

RELEASE_TAG = 'v4-training-expanded'
ASSET_NAME = 'v4_colab_data_expanded.tar.gz'
TAR_PATH = DATA_DIR / ASSET_NAME

if not (DATA_DIR / 'tile_labels_v4b.json').exists():
    print('Downloading V4 training data...')
    if GITHUB_TOKEN:
        import requests
        headers = {'Authorization': f'token {GITHUB_TOKEN}', 'Accept': 'application/vnd.github+json'}
        r = requests.get(f'https://api.github.com/repos/jejuchild/MarsLab/releases/tags/{RELEASE_TAG}', headers=headers)
        assets = r.json().get('assets', [])
        asset = next((a for a in assets if a['name'] == ASSET_NAME), None)
        assert asset, f'Asset {ASSET_NAME} not found in release {RELEASE_TAG}'
        asset_url = asset['url']
        !wget -q --show-progress --header='Authorization: token {GITHUB_TOKEN}' --header='Accept: application/octet-stream' -O {TAR_PATH} {asset_url}
    else:
        url = f'https://github.com/jejuchild/MarsLab/releases/download/{RELEASE_TAG}/{ASSET_NAME}'
        !wget -q --show-progress -O {TAR_PATH} {url}
    
    assert TAR_PATH.exists() and TAR_PATH.stat().st_size > 1_000_000, 'Download failed!'
    print(f'Downloaded: {TAR_PATH.stat().st_size/1e6:.0f}MB')
    print('Extracting...')
    !tar xzf {TAR_PATH} -C {DATA_DIR}
    !rm -f {TAR_PATH}
    print('Done!')
else:
    print('Data already extracted.')

print(f'\nContents of {DATA_DIR}:')
for f in sorted(DATA_DIR.iterdir()):
    if f.is_file():
        print(f'  {f.name}: {f.stat().st_size/1e6:.1f}MB')
    elif f.is_dir():
        n = sum(1 for _ in f.rglob('*.jpg'))
        print(f'  {f.name}/: {n} JPEGs')

## 2. Configuration

In [ ]:
CFG = {
    # Model
    'model_name': 'facebook/dinov2-base',
    'hidden_dim': 768,
    'mola_dim': 25,
    'num_classes': 4,
    'head_hidden': 128,
    'dropout': 0.4,
    
    # LoRA
    'lora_r': 16,
    'lora_alpha': 32,
    'lora_dropout': 0.1,
    'lora_targets': ['query', 'key', 'value'],
    'unfreeze_last_n_blocks': 1,
    
    # Training
    'batch_size': 64,
    'num_epochs': 60,
    'lr_backbone': 1e-4,
    'lr_head': 1e-3,
    'weight_decay': 0.03,
    'warmup_epochs': 5,
    'label_smoothing': 0.1,
    
    # MixUp
    'mixup_alpha': 0.3,
    
    # EMA
    'ema_decay': 0.996,
    
    # AugIN-σ
    'augin_sigma_prob': 0.5,
    
    # Soft-to-hard consistency
    'consistency_weight': 0.5,
    'consistency_rampup': 10,
    'thard': 0.5,
    'tsoft': 0.9,
    
    # Data
    'tile_size': 224,
    'num_workers': 2,
    'class_names': ['LDA', 'LVF', 'CCF', 'OTHER'],
    'class_to_idx': {'LDA': 0, 'LVF': 1, 'CCF': 2, 'OTHER': 3},
    
    # Augmentation
    'aug_hflip': True,
    'aug_vflip': True,
    'aug_rotation': True,
    'aug_random_erasing': 0.2,
    'aug_gaussian_noise': 0.02,
    
    # Paths
    'data_dir': '/content/v4_data',
    'save_dir': '/content/checkpoints',
}

print('V4b FiLM variant:')
print('  MOLA fusion: FiLM conditioning (γ*CLS + β) instead of concatenation')
print('  Head input: 768-dim (modulated CLS) instead of 832-dim (concat)')
print('  Everything else same as V4')

## 3. Dataset with AugIN-σ

In [ ]:
class AugINSigma:
    """
    AugIN-σ from S5Mars (arXiv:2207.01200).
    
    Mars images have concentrated color distributions (low std dev).
    Standard color augmentations hurt Mars SSL performance.
    AugIN-σ swaps std dev between images while preserving mean:
    
    For each channel:
      x_new = (x - μ_x) / σ_x * σ_ref + μ_x
    
    This creates appearance variation without color distribution shift.
    """
    def __init__(self, prob=0.5):
        self.prob = prob
        self._ref_stats = None  # will be set from batch
    
    def set_reference_stats(self, ref_std):
        """Set reference std dev from another image in batch."""
        self._ref_stats = ref_std
    
    def __call__(self, img):
        """Apply to HWC float32 image [0, 1]."""
        if random.random() > self.prob or self._ref_stats is None:
            return img
        
        for c in range(3):
            ch = img[:, :, c]
            mu = ch.mean()
            sigma = ch.std() + 1e-6
            ref_sigma = self._ref_stats[c] + 1e-6
            img[:, :, c] = (ch - mu) / sigma * ref_sigma + mu
        
        return np.clip(img, 0.0, 1.0)


class TrainAugmentation:
    """Mars-safe augmentation pipeline.
    
    NO standard color jitter (hurts Mars images per S5Mars findings).
    Uses AugIN-σ for appearance variation instead.
    """
    def __init__(self, cfg):
        self.hflip = cfg['aug_hflip']
        self.vflip = cfg['aug_vflip']
        self.rotation = cfg['aug_rotation']
        self.erase_prob = cfg.get('aug_random_erasing', 0)
        self.noise_std = cfg.get('aug_gaussian_noise', 0)
        self.augin = AugINSigma(prob=cfg.get('augin_sigma_prob', 0.5))
    
    def __call__(self, img, ref_std=None):
        # AugIN-σ (exchange std dev with reference image)
        if ref_std is not None:
            self.augin.set_reference_stats(ref_std)
            img = self.augin(img)
        
        # Random 90-degree rotation
        if self.rotation:
            k = random.randint(0, 3)
            if k > 0:
                img = np.rot90(img, k=k, axes=(0, 1)).copy()
        
        # Random flips
        if self.hflip and random.random() > 0.5:
            img = np.fliplr(img).copy()
        if self.vflip and random.random() > 0.5:
            img = np.flipud(img).copy()
        
        # Gaussian noise
        if self.noise_std > 0:
            noise = np.random.normal(0, self.noise_std, img.shape).astype(np.float32)
            img = np.clip(img + noise, 0.0, 1.0)
        
        # Random erasing
        if self.erase_prob > 0 and random.random() < self.erase_prob:
            h, w = img.shape[:2]
            eh = random.randint(h // 8, h // 3)
            ew = random.randint(w // 8, w // 3)
            y = random.randint(0, h - eh)
            x = random.randint(0, w - ew)
            img[y:y+eh, x:x+ew] = np.random.uniform(0, 1, (eh, ew, 3)).astype(np.float32)
        
        return img

In [ ]:
class MarsLandformDataset(Dataset):
    """V4 dataset with AugIN-σ support."""
    
    MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32)
    STD = np.array([0.229, 0.224, 0.225], dtype=np.float32)
    
    def __init__(self, tile_labels, split_indices, mola_features, tile_index,
                 data_dir, class_to_idx, transform=None, compute_stats=False):
        self.data_dir = Path(data_dir)
        self.class_to_idx = class_to_idx
        self.transform = transform
        self.mola_features = mola_features
        self.tile_index = tile_index
        
        self.samples = []
        for idx in split_indices:
            t = tile_labels[idx]
            if t['label'] == 'UNLABELED':
                continue
            self.samples.append(t)
        
        # Pre-compute per-image channel std devs for AugIN-σ
        self.image_stds = {}  # image_id → (3,) std per channel
        if compute_stats:
            self._precompute_image_stats()
        
        print(f'  Dataset: {len(self.samples)} samples')
        labels = [s['label'] for s in self.samples]
        for cls_name in sorted(class_to_idx.keys()):
            n = sum(1 for l in labels if l == cls_name)
            print(f'    {cls_name}: {n} ({100*n/len(labels):.1f}%)')
    
    def _precompute_image_stats(self):
        """Compute per-image channel statistics for AugIN-σ."""
        print('  Pre-computing image stats for AugIN-σ...')
        seen = set()
        for s in self.samples:
            img_id = s['image_id']
            if img_id in seen:
                continue
            seen.add(img_id)
            # Load one tile from this image to estimate stats
            img = self._load_tile_image(s)
            if img is not None:
                self.image_stds[img_id] = np.array([
                    img[:,:,0].std(), img[:,:,1].std(), img[:,:,2].std()
                ], dtype=np.float32)
        print(f'  Stats for {len(self.image_stds)} images')
    
    def __len__(self):
        return len(self.samples)
    
    def _load_tile_image(self, sample):
        img_id = sample['image_id']
        tr, tc = sample['tile_row'], sample['tile_col']
        tile_key = f'{img_id}_{tr}_{tc}'
        rel_path = self.tile_index.get(tile_key)
        if rel_path:
            img_path = self.data_dir / rel_path
        else:
            img_path = self.data_dir / 'tiles' / img_id / f'tile_{tr:03d}_{tc:03d}.jpg'
        try:
            img = Image.open(img_path).convert('RGB')
            img = np.array(img, dtype=np.float32) / 255.0
        except Exception:
            img = np.zeros((224, 224, 3), dtype=np.float32)
        return img
    
    def _get_mola(self, sample):
        img_id = sample['image_id']
        tile_key = f"{sample['tile_row']}_{sample['tile_col']}"
        img_mola = self.mola_features.get(img_id, {})
        if tile_key in img_mola:
            return img_mola[tile_key].astype(np.float32)
        return np.zeros(25, dtype=np.float32)
    
    def __getitem__(self, idx):
        sample = self.samples[idx]
        img = self._load_tile_image(sample)
        
        if self.transform:
            # Get reference std from a random different image for AugIN-σ
            ref_std = None
            if self.image_stds:
                ref_idx = random.randint(0, len(self.samples) - 1)
                ref_img_id = self.samples[ref_idx]['image_id']
                ref_std = self.image_stds.get(ref_img_id)
            img = self.transform(img, ref_std=ref_std)
        
        img = (img - self.MEAN) / self.STD
        img_tensor = torch.from_numpy(img.transpose(2, 0, 1))
        mola = torch.from_numpy(self._get_mola(sample))
        label = self.class_to_idx[sample['label']]
        return img_tensor, mola, label

In [ ]:
# Load data
data_dir = Path(CFG['data_dir'])

print('Loading V4 data...')
with open(data_dir / 'tile_labels_v4b.json') as f:
    tile_labels = json.load(f)
with open(data_dir / 'tile_splits_v4b.json') as f:
    splits = json.load(f)
with open(data_dir / 'tile_index.json') as f:
    tile_index = json.load(f)
mola_features = np.load(data_dir / 'mola_features_by_tile.npy', allow_pickle=True).item()

print(f'Total tiles: {len(tile_labels)}')
print(f'Splits - Train: {len(splits["train"])}, Val: {len(splits["val"])}, Test: {len(splits["test"])}')

# Distribution
for split_name in ['train', 'val', 'test']:
    dist = Counter(tile_labels[i]['label'] for i in splits[split_name])
    print(f'  {split_name}: {dict(dist)}')

train_aug = TrainAugmentation(CFG)

print('\nTrain:')
train_ds = MarsLandformDataset(
    tile_labels, splits['train'], mola_features, tile_index,
    data_dir, CFG['class_to_idx'], transform=train_aug, compute_stats=True
)
print('Val:')
val_ds = MarsLandformDataset(
    tile_labels, splits['val'], mola_features, tile_index,
    data_dir, CFG['class_to_idx'], transform=None
)
print('Test:')
test_ds = MarsLandformDataset(
    tile_labels, splits['test'], mola_features, tile_index,
    data_dir, CFG['class_to_idx'], transform=None
)

In [ ]:
# WeightedRandomSampler — oversample minority classes
train_labels = [CFG['class_to_idx'][s['label']] for s in train_ds.samples]
label_counts = Counter(train_labels)
n_samples = len(train_labels)

# Weight per sample = 1 / class_count
class_sample_weights = {cls: n_samples / count for cls, count in label_counts.items()}
sample_weights = [class_sample_weights[l] for l in train_labels]
sampler = WeightedRandomSampler(sample_weights, num_samples=n_samples, replacement=True)

print('Sample weights per class:')
for cls_idx, name in enumerate(CFG['class_names']):
    w = class_sample_weights.get(cls_idx, 0)
    n = label_counts.get(cls_idx, 0)
    print(f'  {name}: weight={w:.2f} (n={n})')

# Dataloaders
train_loader = DataLoader(train_ds, batch_size=CFG['batch_size'], sampler=sampler,
                          num_workers=CFG['num_workers'], pin_memory=True, drop_last=True)
val_loader = DataLoader(val_ds, batch_size=CFG['batch_size'], shuffle=False,
                        num_workers=CFG['num_workers'], pin_memory=True)
test_loader = DataLoader(test_ds, batch_size=CFG['batch_size'], shuffle=False,
                         num_workers=CFG['num_workers'], pin_memory=True)

print(f'\nBatches/epoch - Train: {len(train_loader)}, Val: {len(val_loader)}')

## 4. Model — DINOv2 + LoRA + FiLM(MOLA) (1 block unfrozen)

In [ ]:
from transformers import Dinov2Model
from peft import LoraConfig, get_peft_model


class FiLMLayer(nn.Module):
    """
    Feature-wise Linear Modulation (FiLM).
    
    MOLA features generate per-feature scale (gamma) and shift (beta)
    that modulate the visual CLS token:
        output = gamma * visual_features + beta
    
    This lets terrain context CONDITION how visual features are interpreted,
    rather than being concatenated as a flat vector.
    """
    def __init__(self, mola_dim, visual_dim, hidden_dim=64):
        super().__init__()
        self.mola_encoder = nn.Sequential(
            nn.BatchNorm1d(mola_dim),
            nn.Linear(mola_dim, hidden_dim),
            nn.GELU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.GELU(),
        )
        # Generate gamma (scale) and beta (shift) for each visual feature
        self.gamma_proj = nn.Linear(hidden_dim, visual_dim)
        self.beta_proj = nn.Linear(hidden_dim, visual_dim)
        
        # Initialize gamma close to 1, beta close to 0 (identity init)
        nn.init.ones_(self.gamma_proj.bias)
        nn.init.zeros_(self.gamma_proj.weight)
        nn.init.zeros_(self.beta_proj.bias)
        nn.init.zeros_(self.beta_proj.weight)
    
    def forward(self, visual_features, mola_features):
        """Apply FiLM conditioning.
        visual_features: (B, 768) — DINOv2 CLS token
        mola_features: (B, 25) — terrain features
        Returns: (B, 768) — modulated visual features
        """
        h = self.mola_encoder(mola_features)  # (B, hidden_dim)
        gamma = self.gamma_proj(h)  # (B, 768)
        beta = self.beta_proj(h)    # (B, 768)
        return gamma * visual_features + beta


class MarsLandformNetV4(nn.Module):
    """V4b: FiLM conditioning — MOLA modulates HiRISE visual features."""
    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        
        # DINOv2 backbone
        print('Loading DINOv2-base...')
        self.backbone = Dinov2Model.from_pretrained(cfg['model_name'])
        
        # Apply LoRA
        print('Applying LoRA adapters...')
        lora_config = LoraConfig(
            r=cfg['lora_r'],
            lora_alpha=cfg['lora_alpha'],
            lora_dropout=cfg['lora_dropout'],
            target_modules=cfg['lora_targets'],
            bias='none',
        )
        self.backbone = get_peft_model(self.backbone, lora_config)
        self.backbone.print_trainable_parameters()
        
        # Unfreeze last N blocks
        n_unfreeze = cfg['unfreeze_last_n_blocks']
        if n_unfreeze > 0:
            total_layers = 12
            for i in range(total_layers - n_unfreeze, total_layers):
                layer = self.backbone.base_model.model.encoder.layer[i]
                for param in layer.parameters():
                    param.requires_grad = True
            print(f'Unfroze last {n_unfreeze} transformer blocks')
        
        # FiLM: MOLA conditions visual features (replaces concatenation)
        self.film = FiLMLayer(
            mola_dim=cfg['mola_dim'],      # 25
            visual_dim=cfg['hidden_dim'],    # 768
            hidden_dim=64,
        )
        print('FiLM layer: MOLA(25) -> gamma,beta(768) to modulate CLS token')
        
        # Classification head (input is 768, not 832 — no concatenation)
        self.classifier = nn.Sequential(
            nn.Linear(cfg['hidden_dim'], cfg['head_hidden']),
            nn.BatchNorm1d(cfg['head_hidden']),
            nn.GELU(),
            nn.Dropout(cfg['dropout']),
            nn.Linear(cfg['head_hidden'], cfg['num_classes']),
        )
    
    def forward(self, pixel_values, mola_features):
        outputs = self.backbone(pixel_values=pixel_values)
        cls_token = outputs.last_hidden_state[:, 0]  # (B, 768)
        
        # FiLM: MOLA modulates visual features
        modulated = self.film(cls_token, mola_features)  # (B, 768)
        
        logits = self.classifier(modulated)  # (B, 4)
        return logits

In [ ]:
def load_ssl_lora_weights(model, ssl_path):
    """Load SSL-pretrained LoRA weights."""
    ckpt = torch.load(ssl_path, map_location='cpu')
    ssl_state = ckpt['lora_state_dict']
    mapped = {}
    for ssl_key, tensor in ssl_state.items():
        peft_key = ssl_key.replace('backbone.', '', 1)
        mapped[peft_key] = tensor
    result = model.backbone.load_state_dict(mapped, strict=False)
    loaded = len(mapped) - len(result.unexpected_keys)
    print(f'SSL LoRA weights: {loaded}/{len(mapped)} tensors matched')
    return model


class EMATeacher:
    """
    EMA Teacher for soft-to-hard consistency (S5Mars style).
    The teacher is an EMA of student weights, used to:
    1. Generate pseudo-labels for consistency regularization
    2. Provide smoothed model for validation
    """
    def __init__(self, model, decay=0.996):
        self.decay = decay
        self.shadow = {}
        self.backup = {}
        for name, param in model.named_parameters():
            if param.requires_grad:
                self.shadow[name] = param.data.clone()
    
    def update(self, model):
        for name, param in model.named_parameters():
            if param.requires_grad and name in self.shadow:
                self.shadow[name] = self.decay * self.shadow[name] + (1 - self.decay) * param.data
    
    def apply_shadow(self, model):
        for name, param in model.named_parameters():
            if param.requires_grad and name in self.shadow:
                self.backup[name] = param.data.clone()
                param.data = self.shadow[name]
    
    def restore(self, model):
        for name, param in model.named_parameters():
            if param.requires_grad and name in self.backup:
                param.data = self.backup[name]
        self.backup = {}

In [ ]:
# Build model
model = MarsLandformNetV4(CFG)

ssl_path = data_dir / 'ssl_lora_weights.pt'
if ssl_path.exists():
    model = load_ssl_lora_weights(model, ssl_path)
else:
    print('No SSL weights — training from scratch')

model = model.to(device)

total = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'\nTotal: {total/1e6:.1f}M, Trainable: {trainable/1e6:.2f}M ({100*trainable/total:.1f}%)')

# EMA Teacher
ema = EMATeacher(model, decay=CFG['ema_decay'])

## 5. Training Setup — Soft-to-Hard Consistency Loss

In [ ]:
def soft_to_hard_consistency_loss(student_logits, teacher_logits, thard, tsoft, temperature=1.0):
    """
    Soft-to-hard consistency loss from S5Mars.
    
    For each sample, the teacher generates a pseudo-label:
    - If max teacher prob > tsoft → hard pseudo-label (CE loss)
    - If thard < max teacher prob <= tsoft → soft pseudo-label (KL div loss)
    - If max teacher prob <= thard → ignore (too uncertain)
    
    This avoids error propagation from overconfident wrong pseudo-labels
    while still leveraging uncertain predictions via soft targets.
    """
    with torch.no_grad():
        teacher_probs = F.softmax(teacher_logits / temperature, dim=1)
        max_probs, pseudo_labels = teacher_probs.max(dim=1)
    
    student_log_probs = F.log_softmax(student_logits / temperature, dim=1)
    
    # Hard region: high confidence
    hard_mask = max_probs > tsoft
    # Soft region: medium confidence  
    soft_mask = (max_probs > thard) & (max_probs <= tsoft)
    
    loss = torch.tensor(0.0, device=student_logits.device)
    n_total = 0
    
    if hard_mask.any():
        hard_loss = F.cross_entropy(student_logits[hard_mask], pseudo_labels[hard_mask], reduction='sum')
        loss = loss + hard_loss
        n_total += hard_mask.sum().item()
    
    if soft_mask.any():
        # KL divergence with soft teacher distribution
        soft_loss = F.kl_div(
            student_log_probs[soft_mask],
            teacher_probs[soft_mask],
            reduction='sum'
        ) * (temperature ** 2)
        loss = loss + soft_loss
        n_total += soft_mask.sum().item()
    
    if n_total > 0:
        loss = loss / n_total
    
    return loss, hard_mask.sum().item(), soft_mask.sum().item()


def consistency_rampup(epoch, rampup_length):
    """Sigmoid ramp-up for consistency weight."""
    if epoch >= rampup_length:
        return 1.0
    return math.exp(-5.0 * (1.0 - epoch / rampup_length) ** 2)

In [ ]:
# Class weights for loss
total_train = len(train_ds)
label_dist = Counter([s['label'] for s in train_ds.samples])
class_weights = []
for name in CFG['class_names']:
    count = label_dist.get(name, 1)
    w = math.sqrt(total_train / count)
    class_weights.append(w)
mean_w = sum(class_weights) / len(class_weights)
class_weights = [w / mean_w for w in class_weights]
class_weights_tensor = torch.tensor(class_weights, dtype=torch.float32).to(device)

print('Loss weights:')
for name, w in zip(CFG['class_names'], class_weights):
    print(f'  {name}: {w:.3f}')

criterion = nn.CrossEntropyLoss(
    weight=class_weights_tensor,
    label_smoothing=CFG['label_smoothing']
)

# Optimizer — separate LR for backbone (LoRA + unfrozen block) vs head
backbone_params = [p for n, p in model.named_parameters() if p.requires_grad and 'backbone' in n]
head_params = [p for n, p in model.named_parameters() if p.requires_grad and 'backbone' not in n]

print(f'\nBackbone trainable: {sum(p.numel() for p in backbone_params)/1e6:.2f}M')
print(f'Head trainable: {sum(p.numel() for p in head_params)/1e6:.2f}M')

optimizer = torch.optim.AdamW([
    {'params': backbone_params, 'lr': CFG['lr_backbone']},
    {'params': head_params, 'lr': CFG['lr_head']},
], weight_decay=CFG['weight_decay'])

# Cosine with warmup
total_steps = CFG['num_epochs'] * len(train_loader)
warmup_steps = CFG['warmup_epochs'] * len(train_loader)

def lr_lambda(step):
    if step < warmup_steps:
        return step / max(warmup_steps, 1)
    progress = (step - warmup_steps) / max(total_steps - warmup_steps, 1)
    return max(0.01, 0.5 * (1.0 + math.cos(math.pi * progress)))

scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)
scaler = GradScaler()

print(f'\nTotal steps: {total_steps}, Warmup: {warmup_steps}')

## 6. Training Loop with MixUp + Soft-to-Hard Consistency

In [ ]:
def mixup_data(x, mola, y, alpha=0.3):
    """MixUp: interpolate between random pairs."""
    if alpha <= 0:
        return x, mola, y, y, 1.0
    lam = np.random.beta(alpha, alpha)
    lam = max(lam, 1 - lam)
    batch_size = x.size(0)
    index = torch.randperm(batch_size, device=x.device)
    mixed_x = lam * x + (1 - lam) * x[index]
    mixed_mola = lam * mola + (1 - lam) * mola[index]
    return mixed_x, mixed_mola, y, y[index], lam


def mixup_criterion(criterion, logits, y_a, y_b, lam):
    return lam * criterion(logits, y_a) + (1 - lam) * criterion(logits, y_b)


def train_one_epoch(model, loader, criterion, optimizer, scheduler, scaler, ema,
                    device, cfg, epoch):
    model.train()
    total_loss = 0
    total_cons_loss = 0
    all_preds = []
    all_labels = []
    n_hard = 0
    n_soft = 0
    
    # Consistency ramp-up
    cons_weight = cfg['consistency_weight'] * consistency_rampup(epoch, cfg['consistency_rampup'])
    
    for images, mola, labels in loader:
        images = images.to(device, non_blocking=True)
        mola = mola.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)
        
        # MixUp
        mixed_images, mixed_mola, targets_a, targets_b, lam = mixup_data(
            images, mola, labels, alpha=cfg['mixup_alpha']
        )
        
        optimizer.zero_grad()
        
        with autocast():
            # Student forward (mixed)
            student_logits = model(mixed_images, mixed_mola)
            cls_loss = mixup_criterion(criterion, student_logits, targets_a, targets_b, lam)
            
            # Soft-to-hard consistency with EMA teacher
            cons_loss = torch.tensor(0.0, device=device)
            if cons_weight > 0 and epoch >= 3:  # start after warmup
                ema.apply_shadow(model)
                with torch.no_grad():
                    teacher_logits = model(images, mola)  # unmixed for teacher
                ema.restore(model)
                
                # Student forward on unmixed data for consistency
                student_logits_unmixed = model(images, mola)
                cons_loss, n_h, n_s = soft_to_hard_consistency_loss(
                    student_logits_unmixed, teacher_logits,
                    thard=cfg['thard'], tsoft=cfg['tsoft']
                )
                n_hard += n_h
                n_soft += n_s
            
            loss = cls_loss + cons_weight * cons_loss
        
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()
        
        # EMA update
        ema.update(model)
        
        total_loss += cls_loss.item() * images.size(0)
        total_cons_loss += cons_loss.item() * images.size(0) if cons_weight > 0 else 0
        preds = student_logits.argmax(dim=1).cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(targets_a.cpu().numpy())
    
    avg_loss = total_loss / len(all_labels)
    avg_cons = total_cons_loss / len(all_labels)
    f1 = f1_score(all_labels, all_preds, average='macro')
    acc = sum(p == l for p, l in zip(all_preds, all_labels)) / len(all_labels)
    return avg_loss, avg_cons, f1, acc, n_hard, n_soft


@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss = 0
    all_preds = []
    all_labels = []
    all_probs = []
    
    for images, mola, labels in loader:
        images = images.to(device, non_blocking=True)
        mola = mola.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)
        
        with autocast():
            logits = model(images, mola)
            loss = criterion(logits, labels)
        
        total_loss += loss.item() * images.size(0)
        probs = F.softmax(logits, dim=1).cpu().numpy()
        preds = logits.argmax(dim=1).cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(labels.cpu().numpy())
        all_probs.extend(probs)
    
    avg_loss = total_loss / len(all_labels)
    f1 = f1_score(all_labels, all_preds, average='macro')
    acc = sum(p == l for p, l in zip(all_preds, all_labels)) / len(all_labels)
    return avg_loss, f1, acc, all_preds, all_labels, np.array(all_probs)

In [ ]:
# Mount drive (optional)
try:
    from google.colab import drive
    drive.mount('/content/drive')
    save_dir = Path('/content/drive/MyDrive/marslandform_v4b')
    save_dir.mkdir(parents=True, exist_ok=True)
    CFG['save_dir'] = str(save_dir)
    print(f'Saving to Drive: {save_dir}')
except Exception:
    save_dir = Path(CFG['save_dir'])
    save_dir.mkdir(parents=True, exist_ok=True)
    print(f'Saving locally: {save_dir}')
    print('Download from file browser when done.')

In [ ]:
# ─── TRAINING ──────────────────────────────────────────────────────────────
best_val_f1 = 0.0
patience = 15
patience_counter = 0
history = {
    'train_loss': [], 'train_cons': [], 'train_f1': [],
    'val_loss': [], 'val_f1': [], 'val_f1_ema': [], 'lr': []
}

save_dir = Path(CFG['save_dir'])

print(f'\n{"="*80}')
print(f'Training MarsLandformNet V4b (FiLM) for {CFG["num_epochs"]} epochs')
print(f'  AugIN-σ: prob={CFG["augin_sigma_prob"]}')
print(f'  Consistency: weight={CFG["consistency_weight"]}, rampup={CFG["consistency_rampup"]}ep')
print(f'  thard={CFG["thard"]}, tsoft={CFG["tsoft"]}')
print(f'{"="*80}\n')

for epoch in range(1, CFG['num_epochs'] + 1):
    t0 = time.time()
    
    # Train
    train_loss, train_cons, train_f1, train_acc, n_hard, n_soft = train_one_epoch(
        model, train_loader, criterion, optimizer, scheduler, scaler, ema,
        device, CFG, epoch
    )
    
    # Validate (normal weights)
    val_loss, val_f1, val_acc, val_preds, val_labels, val_probs = evaluate(
        model, val_loader, criterion, device
    )
    
    # Validate with EMA teacher weights
    ema.apply_shadow(model)
    _, val_f1_ema, _, val_preds_ema, val_labels_ema, val_probs_ema = evaluate(
        model, val_loader, criterion, device
    )
    ema.restore(model)
    
    # Use best of EMA vs normal
    use_ema = val_f1_ema > val_f1
    best_this_epoch = max(val_f1, val_f1_ema)
    
    elapsed = time.time() - t0
    current_lr = optimizer.param_groups[0]['lr']
    
    history['train_loss'].append(train_loss)
    history['train_cons'].append(train_cons)
    history['train_f1'].append(train_f1)
    history['val_loss'].append(val_loss)
    history['val_f1'].append(val_f1)
    history['val_f1_ema'].append(val_f1_ema)
    history['lr'].append(current_lr)
    
    improved = ''
    if best_this_epoch > best_val_f1:
        best_val_f1 = best_this_epoch
        patience_counter = 0
        improved = ' ★ BEST'
        
        if use_ema:
            ema.apply_shadow(model)
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'val_f1': best_this_epoch,
            'val_acc': val_acc,
            'cfg': CFG,
            'used_ema': use_ema,
        }, save_dir / 'best_model.pt')
        if use_ema:
            ema.restore(model)
    else:
        patience_counter += 1
    
    ema_tag = f' EMA={val_f1_ema:.4f}' if abs(val_f1_ema - val_f1) > 0.001 else ''
    cons_tag = f' Cons={train_cons:.4f}(H{n_hard}/S{n_soft})' if train_cons > 0 else ''
    print(f'Ep {epoch:3d}/{CFG["num_epochs"]} | '
          f'Train L={train_loss:.4f} F1={train_f1:.4f}{cons_tag} | '
          f'Val L={val_loss:.4f} F1={val_f1:.4f}{ema_tag} | '
          f'LR={current_lr:.2e} | {elapsed:.0f}s{improved}')
    
    if epoch % 5 == 0:
        report_preds = val_preds_ema if use_ema else val_preds
        report_labels = val_labels_ema if use_ema else val_labels
        per_class = f1_score(report_labels, report_preds, average=None)
        for i, name in enumerate(CFG['class_names']):
            print(f'    {name}: F1={per_class[i]:.4f}')
    
    if epoch % 20 == 0:
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'val_f1': val_f1,
            'cfg': CFG,
            'history': history,
        }, save_dir / f'checkpoint_epoch{epoch}.pt')
    
    if patience_counter >= patience:
        print(f'\nEarly stopping at epoch {epoch} (no improvement for {patience} epochs)')
        break

print(f'\n{"="*80}')
print(f'Training complete! Best val F1: {best_val_f1:.4f}')
print(f'{"="*80}')

# Save history
with open(save_dir / 'history_v4b.json', 'w') as f:
    json.dump(history, f)

## 7. Training Curves

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(22, 5))

axes[0].plot(history['train_loss'], label='Train (cls)', linewidth=2)
axes[0].plot(history['val_loss'], label='Val', linewidth=2)
if any(c > 0 for c in history['train_cons']):
    axes[0].plot(history['train_cons'], label='Consistency', linewidth=2, linestyle='--')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss')
axes[0].set_title('Loss'); axes[0].legend(); axes[0].grid(True, alpha=0.3)

axes[1].plot(history['train_f1'], label='Train', linewidth=2)
axes[1].plot(history['val_f1'], label='Val', linewidth=2)
axes[1].plot(history['val_f1_ema'], label='Val (EMA)', linewidth=2, linestyle='--')
axes[1].axhline(y=0.8, color='r', linestyle='--', alpha=0.5, label='Target (0.8)')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Macro F1')
axes[1].set_title('F1 Score'); axes[1].legend(); axes[1].grid(True, alpha=0.3)

axes[2].plot(history['lr'], linewidth=2, color='green')
axes[2].set_xlabel('Epoch'); axes[2].set_ylabel('LR')
axes[2].set_title('Learning Rate'); axes[2].grid(True, alpha=0.3)

# Overfitting gap
gap = [t - v for t, v in zip(history['train_f1'], history['val_f1'])]
axes[3].plot(gap, linewidth=2, color='orange')
axes[3].axhline(y=0, color='k', linestyle='-', alpha=0.3)
axes[3].set_xlabel('Epoch'); axes[3].set_ylabel('Train F1 - Val F1')
axes[3].set_title('Overfit Gap'); axes[3].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(save_dir / 'training_curves_v4b.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. Test Evaluation

In [ ]:
# Load best model
best_ckpt = torch.load(save_dir / 'best_model.pt', map_location=device)
model.load_state_dict(best_ckpt['model_state_dict'])
print(f'Best model: epoch {best_ckpt["epoch"]}, val F1={best_ckpt["val_f1"]:.4f}, EMA={best_ckpt.get("used_ema", False)}')

test_loss, test_f1, test_acc, test_preds, test_labels, test_probs = evaluate(
    model, test_loader, criterion, device
)

print(f'\n{"="*60}')
print(f'  TEST SET RESULTS — MarsLandformNet V4b (FiLM)')
print(f'{"="*60}')
print(f'  Macro F1:  {test_f1:.4f}   {"✓ TARGET MET" if test_f1 >= 0.8 else "✗ Below target (0.8)"}')
print(f'  Accuracy:  {test_acc:.4f}')
print(f'{"="*60}\n')
print(classification_report(test_labels, test_preds,
                            target_names=CFG['class_names'], digits=4))

In [ ]:
cm = confusion_matrix(test_labels, test_preds)
cm_pct = cm.astype('float') / cm.sum(axis=1, keepdims=True) * 100

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=CFG['class_names'], yticklabels=CFG['class_names'], ax=ax1)
ax1.set_xlabel('Predicted'); ax1.set_ylabel('True')
ax1.set_title('Confusion Matrix (Counts)')

sns.heatmap(cm_pct, annot=True, fmt='.1f', cmap='Blues',
            xticklabels=CFG['class_names'], yticklabels=CFG['class_names'], ax=ax2)
ax2.set_xlabel('Predicted'); ax2.set_ylabel('True')
ax2.set_title('Confusion Matrix (%)')

plt.tight_layout()
plt.savefig(save_dir / 'confusion_matrix_v4b.png', dpi=150, bbox_inches='tight')
plt.show()

## 9. Confidence Analysis

In [ ]:
# Per-class confidence distributions
test_probs_np = np.array(test_probs)
test_labels_np = np.array(test_labels)
test_preds_np = np.array(test_preds)

fig, axes = plt.subplots(1, 4, figsize=(20, 4))
for cls_idx, cls_name in enumerate(CFG['class_names']):
    mask = test_labels_np == cls_idx
    correct_mask = mask & (test_preds_np == cls_idx)
    wrong_mask = mask & (test_preds_np != cls_idx)
    
    if correct_mask.any():
        axes[cls_idx].hist(test_probs_np[correct_mask, cls_idx], bins=30, alpha=0.7,
                          label=f'Correct ({correct_mask.sum()})', color='green')
    if wrong_mask.any():
        axes[cls_idx].hist(test_probs_np[wrong_mask, cls_idx], bins=30, alpha=0.7,
                          label=f'Wrong ({wrong_mask.sum()})', color='red')
    axes[cls_idx].set_title(f'{cls_name}')
    axes[cls_idx].set_xlabel('P(class)')  
    axes[cls_idx].legend(fontsize=8)
    axes[cls_idx].grid(True, alpha=0.3)

plt.suptitle('Prediction Confidence by Class', fontsize=14)
plt.tight_layout()
plt.savefig(save_dir / 'confidence_v4b.png', dpi=150, bbox_inches='tight')
plt.show()

## 10. Export for MarsLab

In [ ]:
deploy_state = {
    'model_state_dict': model.state_dict(),
    'cfg': CFG,
    'class_names': CFG['class_names'],
    'class_to_idx': CFG['class_to_idx'],
    'test_f1': test_f1,
    'test_acc': test_acc,
    'epoch': best_ckpt['epoch'],
    'version': 'v4b-film',
}
deploy_path = save_dir / 'marslandform_v4b_deploy.pt'
torch.save(deploy_state, deploy_path)
print(f'Deploy checkpoint: {deploy_path} ({deploy_path.stat().st_size/1e6:.1f}MB)')
print(f'Test F1={test_f1:.4f}, Acc={test_acc:.4f}')
print(f'\nDownload from file browser or Google Drive: {CFG["save_dir"]}')

## 11. Error Analysis

In [ ]:
misclassified = [(i, p, t) for i, (p, t) in enumerate(zip(test_preds, test_labels)) if p != t]
print(f'Misclassified: {len(misclassified)}/{len(test_preds)} ({100*len(misclassified)/len(test_preds):.1f}%)')

# Show most confident errors
errors_with_conf = []
for idx, pred, true in misclassified:
    conf = test_probs[idx][pred]  # confidence in wrong prediction
    errors_with_conf.append((idx, pred, true, conf))
errors_with_conf.sort(key=lambda x: -x[3])  # most confident errors first

print(f'\nMost confident errors:')
for idx, pred, true, conf in errors_with_conf[:10]:
    print(f'  #{idx}: True={CFG["class_names"][true]}, Pred={CFG["class_names"][pred]} (conf={conf:.3f})')

fig, axes = plt.subplots(3, 5, figsize=(15, 9))
random.shuffle(misclassified)
for i, ax in enumerate(axes.flat):
    if i >= len(misclassified):
        ax.axis('off'); continue
    idx, pred, true = misclassified[i]
    img_t, _, _ = test_ds[idx]
    mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
    std = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)
    img_display = (img_t * std + mean).clamp(0, 1).permute(1, 2, 0).numpy()
    ax.imshow(img_display)
    ax.set_title(f'T:{CFG["class_names"][true]} P:{CFG["class_names"][pred]}', fontsize=9, color='red')
    ax.axis('off')
plt.suptitle('Misclassified Tiles (Random Sample)', fontsize=14)
plt.tight_layout()
plt.savefig(save_dir / 'errors_v4b.png', dpi=150, bbox_inches='tight')
plt.show()